In [2]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))

API key loaded: True


In [3]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path(".")  # current folder

# Load core files
races = pd.read_csv(DATA_DIR / "races.csv")
results = pd.read_csv(DATA_DIR / "results.csv")
lap_times = pd.read_csv(DATA_DIR / "lap_times.csv")
drivers = pd.read_csv(DATA_DIR / "drivers.csv")
constructors = pd.read_csv(DATA_DIR / "constructors.csv")
circuits = pd.read_csv(DATA_DIR / "circuits.csv")

print("Loaded ✅")
print("races:", races.shape)
print("results:", results.shape)
print("lap_times:", lap_times.shape)
print("drivers:", drivers.shape)
print("constructors:", constructors.shape)
print("circuits:", circuits.shape)

races.head(2)

Loaded ✅
races: (1149, 18)
results: (27238, 18)
lap_times: (615738, 6)
drivers: (864, 9)
constructors: (212, 5)
circuits: (77, 9)


,raceId,year,round,circuitId,name,date,time,url,fp1_date,fp1_time,fp2_date,fp2_time,fp3_date,fp3_time,quali_date,quali_time,sprint_date,sprint_time
0,1,2009,1,1,Australian Grand Prix,2009-03-29,06:00:00,http://en.wikipedia.org/wiki/2009_Australian_G...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
1,2,2009,2,2,Malaysian Grand Prix,2009-04-05,09:00:00,http://en.wikipedia.org/wiki/2009_Malaysian_Gr...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


In [4]:
def get_circuit_name(circuitId):
    circuit = circuits[circuits["circuitId"] == circuitId]
    if not circuit.empty:
        return circuit.iloc[0]["name"]
    return "Unknown Circuit"


def get_constructor_name(constructorId):
    constructor = constructors[constructors["constructorId"] == constructorId]
    if not constructor.empty:
        return constructor.iloc[0]["name"]
    return "Unknown Constructor"

In [5]:
# Keep only needed columns
races_small = races[["raceId", "year", "round", "circuitId", "name"]].copy()
results_small = results[["raceId", "driverId", "constructorId", "grid", "positionOrder"]].copy()
laps_small = lap_times[["raceId", "driverId", "lap", "milliseconds"]].copy()

# Merge laps + race info + result info
df = laps_small.merge(races_small, on="raceId", how="left") \
               .merge(results_small, on=["raceId", "driverId"], how="left")

# Convert to numeric safely (invalid → NaN)
df["grid"] = pd.to_numeric(df["grid"], errors="coerce")
df["constructorId"] = pd.to_numeric(df["constructorId"], errors="coerce")

# Now drop rows where they became NaN
df = df.dropna(subset=["constructorId", "grid"])

# Convert to int
df["grid"] = df["grid"].astype(int)
df["constructorId"] = df["constructorId"].astype(int)

In [6]:
print("Final ML table shape:", df.shape)
df.head()

Final ML table shape: (614672, 11)


,raceId,driverId,lap,milliseconds,year,round,circuitId,name,constructorId,grid,positionOrder
0,841,20,1,98109,2011,1,1,Australian Grand Prix,9,1,1
1,841,20,2,93006,2011,1,1,Australian Grand Prix,9,1,1
2,841,20,3,92713,2011,1,1,Australian Grand Prix,9,1,1
3,841,20,4,92803,2011,1,1,Australian Grand Prix,9,1,1
4,841,20,5,92342,2011,1,1,Australian Grand Prix,9,1,1


In [7]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

features = ["year", "round", "circuitId", "constructorId", "grid", "lap"]
X = df[features]
y = df["milliseconds"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

pred = model.predict(X_test)

mae = mean_absolute_error(y_test, pred)

print("MAE (ms):", round(mae, 2))
print("MAE (seconds):", round(mae/1000, 3))

MAE (ms): 2697.02
MAE (seconds): 2.697


In [8]:
sample = pd.DataFrame([{
    "year": 2011,
    "round": 1,
    "circuitId": 1,
    "constructorId": 9,
    "grid": 1,
    "lap": 20
}])

pred_ms = model.predict(sample)[0]
print("Predicted lap time:", round(pred_ms/1000, 3), "seconds")

Predicted lap time: 91.187 seconds


In [9]:
def simulate_strategy(model, base_input, pit_lap_change):
    """
    base_input: dict with year, round, circuitId, constructorId, grid, lap
    pit_lap_change: +3 means pitting 3 laps later, -3 means earlier
    """
    base = pd.DataFrame([base_input])
    base_pred = model.predict(base)[0]

    new = base_input.copy()
    new["lap"] = new["lap"] + pit_lap_change
    new_df = pd.DataFrame([new])
    new_pred = model.predict(new_df)[0]

    delta = (new_pred - base_pred) / 1000

    return {
        "original_time_s": round(base_pred/1000, 3),
        "new_time_s": round(new_pred/1000, 3),
        "delta_seconds": round(delta, 3)
    }


In [10]:
scenario = {
    "year": 2011,
    "round": 1,
    "circuitId": 1,
    "constructorId": 9,
    "grid": 1,
    "lap": 20
}

print(simulate_strategy(model, scenario, pit_lap_change=-3))
print(simulate_strategy(model, scenario, pit_lap_change=+3))


{'original_time_s': 91.187, 'new_time_s': 92.274, 'delta_seconds': 1.087}
{'original_time_s': 91.187, 'new_time_s': 91.244, 'delta_seconds': 0.057}


In [11]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("Key loaded:", OPENAI_API_KEY[:6], "*****")


Key loaded: sk-pro *****


In [12]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

def explain_strategy(original, new, delta):
    prompt = f"""
You are a Formula 1 race strategy engineer.

Original predicted lap time: {original:.3f} seconds
New predicted lap time: {new:.3f} seconds
Time difference: {delta:.3f} seconds

Explain what this means for race strategy in simple terms.
Should the driver pit earlier or later? Why?
"""

    response = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )

    return response.output[0].content[0].text


In [13]:
res_early = simulate_strategy(model, scenario, -3)
res_late = simulate_strategy(model, scenario, +3)

print("EARLY PIT:\n", explain_strategy(
    res_early["original_time_s"],
    res_early["new_time_s"],
    res_early["delta_seconds"]
))

print("\nLATE PIT:\n", explain_strategy(
    res_late["original_time_s"],
    res_late["new_time_s"],
    res_late["delta_seconds"]
))


EARLY PIT:
 The new predicted lap time being **1.087 seconds slower** than originally expected means that the car will take longer to complete a lap than initially planned.

**What this means for race strategy:**
- The car will lose more time on the track than we thought.
- This could be because of factors like tire degradation, fuel load, or track conditions.

**Should the driver pit earlier or later?**
- **Pit earlier.** 

**Why?**
- Since lap times are slower, it’s better to pit earlier to get onto fresher tires sooner.
- Fresh tires will help the driver go faster and recover some time.
- If we wait too long, the driver will continue to lose time each lap on worn tires, making it harder to catch up.

In simple terms: Because the car will be slower on track, it’s smart to stop sooner, get fresh tires, and drive faster laps to make up for the lost time.

LATE PIT:
 The lap time has increased slightly from 91.187 seconds to 91.244 seconds, which means the driver is expected to be about

In [14]:
from IPython.display import display, Markdown

In [15]:
res_early = simulate_strategy(model, scenario, -3)
res_late = simulate_strategy(model, scenario, +3)

early_md = explain_strategy(
    res_early["original_time_s"],
    res_early["new_time_s"],
    res_early["delta_seconds"]
)

late_md = explain_strategy(
    res_late["original_time_s"],
    res_late["new_time_s"],
    res_late["delta_seconds"]
)

display(Markdown("## 🟥 EARLY PIT ANALYSIS\n" + early_md))
display(Markdown("## 🟩 LATE PIT ANALYSIS\n" + late_md))


## 🟥 EARLY PIT ANALYSIS
The new predicted lap time is slower by 1.087 seconds compared to the original prediction (92.274 seconds vs. 91.187 seconds). This means the track conditions, car performance, or tire condition is expected to make each lap take longer than initially thought.

**What this means for race strategy:**

- **Slower laps mean** the driver will lose more time on the track per lap.
- **Pitting earlier** might now be better if the fresh tires allow faster lap times later and help reduce overall time loss.
- **Pitting later** might be worse because the slower laps on worn tires will cost more time.

**Advice:**

Because the laps are predicted to be slower, the driver should consider pitting **earlier** to switch to fresher tires and improve lap times sooner, minimizing the time lost per lap due to slower pace. This can help recover some of the lost time from the slower predicted lap times.

## 🟩 LATE PIT ANALYSIS
The new predicted lap time is **0.057 seconds slower** than the original prediction (91.244 seconds vs. 91.187 seconds). 

In simple terms, this means the car is expected to be slightly slower on track than originally thought.

**What does this mean for race strategy?**  
- If the lap times are slower, the driver will lose a little bit more time each lap.
- Pitting earlier might be better to try to get fresh tires sooner, use the faster tire performance longer, and gain back time.
- Pitting later means running slower laps longer and losing even more time.

**Conclusion:**
The driver should **pit earlier** because the slower lap times suggest that staying out longer on worn tires will cost more time. Fresh tires sooner can help gain time back on track with faster laps.

In [16]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

client = OpenAI(api_key=OPENAI_API_KEY)

In [17]:
def explain_strategy_md(original, new, delta, context_title="Strategy Note"):
    prompt = f"""
You are a Formula 1 race strategy engineer. Write in clean Markdown.

### Context
{context_title}

### Model outputs
- Original predicted lap time: {original:.3f} s
- New predicted lap time: {new:.3f} s
- Delta: {delta:.3f} s (positive means slower)

### Requirements
- Start with a short 1–2 line summary.
- Then a bullet list: What it implies / risk / recommendation.
- End with a clear recommendation: **Pit earlier** or **Pit later** (or **No strong change**).
"""
    resp = client.responses.create(
        model="gpt-4.1-mini",
        input=prompt
    )
    return resp.output[0].content[0].text


In [18]:
import gradio as gr
import pandas as pd

def run_analysis(year, round_no, circuitId, constructorId, grid, lap, pit_change):
    scenario = {
        "year": int(float(year)),
        "round": int(float(round_no)),
        "circuitId": int(float(circuitId)),
        "constructorId": int(float(constructorId)),
        "grid": int(float(grid)),
        "lap": int(float(lap)),
        "pit_change":int(float(pit_change))

    }

    cname = get_circuit_name(scenario["circuitId"])
    tname = get_constructor_name(scenario["constructorId"])


    res = simulate_strategy(model, scenario, int(pit_change))

    title = f"Scenario: Y{scenario['year']} R{scenario['round']} C{scenario['circuitId']} | Team {scenario['constructorId']} | Grid {scenario['grid']} | Lap {scenario['lap']} | Δlap {pit_change}"
    md = explain_strategy_md(res["original_time_s"], res["new_time_s"], res["delta_seconds"], context_title=title)

    # Also return a small structured summary
    summary = f"""
### 📌 Numbers
- Original: **{res['original_time_s']} s**
- New: **{res['new_time_s']} s**
- Delta: **{res['delta_seconds']} s**
"""
    return summary, md

def ask_engineer(user_question, year, round_no, circuitId, constructorId, grid, lap):
    scenario = {
        "year": int(year),
        "round": int(round_no),
        "circuitId": int(circuitId),
        "constructorId": int(constructorId),
        "grid": int(grid),
        "lap": int(lap)
    }

    # We’ll compute two quick reference sims for context
    res_early = simulate_strategy(model, scenario, -3)
    res_late  = simulate_strategy(model, scenario, +3)

    prompt = f"""
You are an F1 race engineer. Answer in Markdown.

### Current scenario
{scenario}

### Model reference (pit timing sensitivity)
- Pit earlier (-3 laps): delta {res_early['delta_seconds']} s, new lap {res_early['new_time_s']} s
- Pit later (+3 laps): delta {res_late['delta_seconds']} s, new lap {res_late['new_time_s']} s

### User question
{user_question}

Rules:
- Use the model deltas above as evidence.
- If the user asks something not supported by the inputs, say what additional data is needed (tyres, weather, safety car, etc.).
- Keep it concise but helpful.
"""
    resp = client.responses.create(model="gpt-4.1-mini", input=prompt)
    return resp.output[0].content[0].text


with gr.Blocks() as demo:
    gr.Markdown("# 🏁 F1 Race Intelligence Copilot\nML lap-time model + LLM race-engineer explanations (Markdown).")

    with gr.Row():
        year = gr.Number(value=2011, label="Year")
        round_no = gr.Number(value=1, label="Round")
        circuitId = gr.Number(value=1, label="Circuit ID")
        constructorId = gr.Number(value=9, label="Constructor ID")
        grid = gr.Number(value=1, label="Grid Position")
        lap = gr.Number(value=20, label="Lap")

    pit_change = gr.Slider(minimum=-10, maximum=10, step=1, value=-3, label="Pit timing change (laps)")

    btn = gr.Button("Run Strategy Analysis")

    out_summary = gr.Markdown()
    out_md = gr.Markdown()

    btn.click(
        fn=run_analysis,
        inputs=[year, round_no, circuitId, constructorId, grid, lap, pit_change],
        outputs=[out_summary, out_md]
    )

    gr.Markdown("---")
    gr.Markdown("## 💬 Ask the Race Engineer")

    question = gr.Textbox(placeholder="e.g., Should we undercut or overcut here? What’s the safer call?", label="Your question")
    ask_btn = gr.Button("Ask")
    answer_md = gr.Markdown()

    ask_btn.click(
        fn=ask_engineer,
        inputs=[question, year, round_no, circuitId, constructorId, grid, lap],
        outputs=answer_md
    )

demo.launch()


* Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


In [19]:
import gradio as gr
import pandas as pd

def run_analysis(year, round_no, circuitId, constructorId, grid, lap, pit_change):
    scenario = {
        "year": int(float(year)),
        "round": int(float(round_no)),
        "circuitId": int(float(circuitId)),
        "constructorId": int(float(constructorId)),
        "grid": int(float(grid)),
        "lap": int(float(lap)),
        "pit_change":int(float(pit_change))

    }

  # Get human-readable names
    cname = get_circuit_name(scenario["circuitId"])
    tname = get_constructor_name(scenario["constructorId"])

    # Run simulation
    res = simulate_strategy(model, scenario, scenario["pit_change"])

    title = (
        f"Scenario: **{scenario['year']} Round {scenario['round']}** | "
        f"**{cname}** | Team: **{tname}** | "
        f"Grid: **{scenario['grid']}** | Lap: **{scenario['lap']}** | "
        f"Δlap: **{scenario['pit_change']}**"
    )

    md = explain_strategy_md(
        res["original_time_s"],
        res["new_time_s"],
        res["delta_seconds"],
        context_title=title
    )

    summary = f"""
### 📌 Numbers
- Original: **{res['original_time_s']} s**
- New: **{res['new_time_s']} s**
- Delta: **{res['delta_seconds']} s**
"""

    return summary, md


In [20]:
demo.launch()

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----

To create a public link, set `share=True` in `launch()`.


In [21]:
def compare_two_teams(year, round_no, circuitId, grid, lap, pit_change, teamA, teamB):
    # Cast Gradio Numbers safely
    year = int(float(year))
    round_no = int(float(round_no))
    circuitId = int(float(circuitId))
    grid = int(float(grid))
    lap = int(float(lap))
    pit_change = int(float(pit_change))
    teamA = int(float(teamA))
    teamB = int(float(teamB))

    cname = get_circuit_name(circuitId)
    A = get_constructor_name(teamA)
    B = get_constructor_name(teamB)

    baseA = {"year": year, "round": round_no, "circuitId": circuitId, "constructorId": teamA, "grid": grid, "lap": lap}
    baseB = {"year": year, "round": round_no, "circuitId": circuitId, "constructorId": teamB, "grid": grid, "lap": lap}

    resA = simulate_strategy(model, baseA, pit_change)
    resB = simulate_strategy(model, baseB, pit_change)

    winner = A if resA["new_time_s"] < resB["new_time_s"] else B

    summary = f"""
## 🆚 Team vs Team Comparison
**Circuit:** {cname}  
**Scenario:** Year {year}, Round {round_no}, Grid {grid}, Lap {lap}, Δlap {pit_change}

| Team | Original (s) | New (s) | Delta (s) |
|---|---:|---:|---:|
| {A} | {resA['original_time_s']:.3f} | {resA['new_time_s']:.3f} | {resA['delta_seconds']:.3f} |
| {B} | {resB['original_time_s']:.3f} | {resB['new_time_s']:.3f} | {resB['delta_seconds']:.3f} |

✅ **Predicted faster team after change:** **{winner}**
"""

    prompt = f"""
You are an F1 performance engineer. Explain in Markdown.

Circuit: {cname}
Year {year}, Round {round_no}, Grid {grid}, Lap {lap}, Δlap {pit_change}

Team A: {A} → new {resA['new_time_s']:.3f}s (Δ {resA['delta_seconds']:.3f}s)
Team B: {B} → new {resB['new_time_s']:.3f}s (Δ {resB['delta_seconds']:.3f}s)

Explain:
- Who benefits more and why (based on deltas)
- What it means for strategy
- One limitation (tyres/traffic/weather not modeled)
"""
    resp = client.responses.create(model="gpt-4.1-mini", input=prompt)
    explanation = resp.output[0].content[0].text

    return summary, explanation


In [22]:
# Build dropdown choices from constructors.csv
constructors = pd.read_csv(DATA_DIR / "constructors.csv")

constructor_choices = [
    (row["name"], int(row["constructorId"]))
    for _, row in constructors.iterrows()
]

print("Example choices:", constructor_choices[:5])


Example choices: [('McLaren', 1), ('BMW Sauber', 2), ('Williams', 3), ('Renault', 4), ('Toro Rosso', 5)]


In [23]:
import gradio as gr
gr.close_all()

In [24]:
import gradio as gr

def compare_two_teams_ui(year, round_no, circuitId, grid, lap, pit_change, teamA, teamB):
    # just call your existing function
    return compare_two_teams(year, round_no, circuitId, grid, lap, pit_change, teamA, teamB)

gr.close_all()

with gr.Blocks() as app_compare:
    gr.Markdown("# 🏁 F1 Race Intelligence Copilot — Compare Teams")

    with gr.Row():
        year = gr.Number(value=2011, label="Year")
        round_no = gr.Number(value=1, label="Round")
        circuitId = gr.Number(value=1, label="Circuit ID")
        grid = gr.Number(value=1, label="Grid Position")
        lap = gr.Number(value=20, label="Lap")

    pit_change = gr.Slider(-10, 10, value=-3, step=1, label="Pit timing change (laps)")

    teamA = gr.Dropdown(choices=constructor_choices, value=9, label="Team A")
    teamB = gr.Dropdown(choices=constructor_choices, value=6, label="Team B")

    btn = gr.Button("Compare Teams")

    out1 = gr.Markdown()
    out2 = gr.Markdown()

    btn.click(
        fn=compare_two_teams_ui,
        inputs=[year, round_no, circuitId, grid, lap, pit_change, teamA, teamB],
        outputs=[out1, out2]
    )

app_compare.launch(share=True, show_error=True)



* Running on local URL:  http://127.0.0.1:7861
* Running on public URL: https://677f3414013f6a3182.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [25]:
app_compare.launch(share=True, show_error=True)

Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
* Running on public URL: https://677f3414013f6a3182.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [26]:
import pandas as pd

lap_times = pd.read_csv("lap_times.csv")

print("Total lap records:", len(lap_times))
print(lap_times.head())

Total lap records: 615738
   raceId  driverId  lap  position      time  milliseconds
0     841        20    1         1  1:38.109         98109
1     841        20    2         1  1:33.006         93006
2     841        20    3         1  1:32.713         92713
3     841        20    4         1  1:32.803         92803
4     841        20    5         1  1:32.342         92342
